In [ ]:
__author__ = "Kay Töpfer, Matthias Groh"
__organization__="SE G QPI QM P"
__copyright__ = "Copyright 2020, Siemens Gas and Power GmbH & Co. KG"
__email__ = "kay.toepfer@siemens.com, groh.matthias@siemens.com"

In [ ]:
import numpy as np
import pandas as pd
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def get_converted_data(data,xData,yData,trainPercShare,randomShuffle,scaling):
    data=data.sample(frac=1, random_state=1) #Shuffle data
    if randomShuffle==True:
        data=data.sample(frac=1) #Shuffle data random
    trainShare=round(len(data)*trainPercShare/100) #Calculate Number of Samples for trainings data
    #Define trainings and test data
    trainData=data[:trainShare]
    testData=data[trainShare:]
    xTrain=trainData.loc[:,xData].values
    yTrain=trainData[yData].values
    xTest=testData.loc[:,xData].values
    yTest=testData[yData].values
    #Scale data
    scaler=None
    if scaling==True:
        try:
            scaler=StandardScaler()
            scaler.fit(xTrain)
            xTrain=scaler.transform(xTrain)
        except:
            scaler.fit(xTest)
            xTest=scaler.transform(xTest)
        try:
            xTest=scaler.transform(xTest)
        except: 
            pass
    return(xTrain,yTrain,xTest,yTest,scaler)

def get_hidden_layer_structure(hiddenLayers,neurons,exactHiddenLayerStructure):
    if exactHiddenLayerStructure==False:
        exactHiddenLayerStructure=[] #Create List for Hidden Layer structure
        while hiddenLayers!=0: #Appends for each Hidden Layer the number of Neurons
            exactHiddenLayerStructure.append(neurons) 
            hiddenLayers-=1
    return(tuple(exactHiddenLayerStructure)) #Convert List into Tuple

def update_result_2outputs(input_,output1,output2,resultEx):
    if input_ in resultEx[resultEx.columns[0]]:
        resultEx=resultEx.drop([input_])
    newRow=pd.Series([input_,output1,output2],index=resultEx.columns)
    resultEx=pd.concat([pd.DataFrame([newRow],index=[input_]),resultEx])
    return(resultEx)

def update_result_3outputs(input_,output1,output2,output3,resultEx):
    if input_ in resultEx[resultEx.columns[0]]:
        resultEx=resultEx.drop([input_])
    newRow=pd.Series([input_,output1,output2,output3],index=resultEx.columns)
    resultEx=pd.concat([pd.DataFrame([newRow],index=[input_]),resultEx])
    return(resultEx)

def create_decision_tree(data,xData,yData,maxDepth,trainPercShare,randomShuffle,randomState):
    convertedData = get_converted_data(data,xData,yData,trainPercShare,randomShuffle,False)
    xTrain=convertedData[0]
    yTrain=convertedData[1]
    xTest=convertedData[2]
    yTest=convertedData[3]
    #Calculate Decision Tree
    clf=DecisionTreeClassifier(criterion='entropy',max_depth=maxDepth,random_state=randomState).fit(xTrain,yTrain)
    #Calculate scores
    testScore=clf.score(xTest,yTest)
    trainScore=clf.score(xTrain,yTrain)
    #Print scores
    print('Test Accuracy: '+str(round(testScore*100,1))+'%')
    print('Train Accuracy: '+str(round(trainScore*100,1))+'%')
    return(maxDepth,testScore,trainScore,clf,[xTrain,yTrain,xTest,yTest])

def create_neural_network(data,xData,yData,learningRate,hiddenLayers,neurons,activationFkt,solvingAlgo,maxEpochs,trainPercShare,randomShuffle,randomState,exactHiddenLayerStructure):
    #Get converted data
    convertedData = get_converted_data(data,xData,yData,trainPercShare,randomShuffle,True)
    xTrain=convertedData[0]
    yTrain=convertedData[1]
    xTest=convertedData[2]
    yTest=convertedData[3]
    #Get Hidden Layer structure
    hiddenLayerStructure=get_hidden_layer_structure(hiddenLayers,neurons,exactHiddenLayerStructure)
    #Calculate Decision Tree
    clf=MLPClassifier(hidden_layer_sizes=hiddenLayerStructure,activation=activationFkt,solver=solvingAlgo,learning_rate_init=learningRate,max_iter=maxEpochs,random_state=randomState).fit(xTrain,yTrain)
    #Calculate scores
    testScore=clf.score(xTest,yTest)
    trainScore=clf.score(xTrain,yTrain)
    #Print scores
    print('Test Accuracy: '+str(round(testScore*100,1))+'%')
    print('Train Accuracy: '+str(round(trainScore*100,1))+'%')
    return(learningRate,hiddenLayers,testScore,trainScore,clf,[xTrain,yTrain,xTest,yTest])

def create_plot(resultEx,logarithmic):
    style='-'
    if len(resultEx)==1:
        style='o'
    resultEx=resultEx.sort_values(by=[resultEx.columns[0]])
    if logarithmic==True:
        resultEx[resultEx.columns[0]]=np.log10(resultEx[resultEx.columns[0]].astype('float'))
    plt.plot(resultEx[resultEx.columns[0]],resultEx[resultEx.columns[1]],style,label=resultEx.columns[1])
    plt.plot(resultEx[resultEx.columns[0]],resultEx[resultEx.columns[2]],style,label=resultEx.columns[2])
    plt.grid(True)
    plt.ylim(0, 105)
    plt.xlabel(resultEx.columns[0])
    plt.ylabel('Accuracy [%]')
    if logarithmic==True:
        resultEx[resultEx.columns[0]]=np.power(10,resultEx[resultEx.columns[0]])
        plt.xlabel(resultEx.columns[0]+' [*10^x]')
    plt.legend()
    plt.show()
    
def show_trees(resultEx):
    resultEx=resultEx.sort_values(by=['Max Depth'])
    for row in resultEx.itertuples(index=True, name='Pandas'):
        print('Max Depth: '+str(row[1]))
        plt.figure(dpi=100)
        treeInfo=tree.plot_tree(row[4])
        plt.show()

def update_NN(model,xUpdate,yUpdate,xTrain,yTrain,xTest,yTest,iterations):
    testScoreBeforeUpdate=model.score(xTest,yTest)
    trainScoreBeforeUpdate=model.score(xTrain,yTrain)
    for i in range(0,iterations):
        modelUpdated=model.partial_fit(xUpdate,yUpdate)
    testScoreAfterUpdate=model.score(xTest,yTest)
    trainScoreAfterUpdate=model.score(xTrain,yTrain)
    print('Test accuracy before updating the model: '+str(round(testScoreBeforeUpdate*100,1))+'%')
    #print('Train accuracy before updating the model: '+str(round(trainScoreBeforeUpdate*100,1))+'%')
    print()
    print('Test accuracy after updating the model:  '+str(round(testScoreAfterUpdate*100,1))+'%')
    #print('Train accuracy after updating the model: '+str(round(trainScoreAfterUpdate*100,1))+'%')
    return(modelUpdated)

In [ ]:
resultEx1=pd.DataFrame(columns=['Max Depth','Test Accuracy','Train Accuracy','clf'])
resultEx2=pd.DataFrame(columns=['Max Depth','Test Accuracy','Train Accuracy','clf'])
resultEx31=pd.DataFrame(columns=['Learning Rate','Test Accuracy','Train Accuracy'])
resultEx32=pd.DataFrame(columns=['Hidden Layers','Test Accuracy','Train Accuracy'])

## Exercise 1: Decision Trees  
### Explore the Parameter **Max Depth** with **Iris** Dataset

In [ ]:
###################################################### Specify data ##########################################################################

dataIris=pd.read_csv('Iris.csv')
xDataIris=['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm']
yDataIris='Species'

In [ ]:
####################################################### For profis ###########################################################################

trainPercShareIris=50 #Number between 0 and 100
randomShuffle=False #True (or False)
randomState=1 #None or Number

In [ ]:
#################################################### Specify parameters ######################################################################

maxDepth=1 #Number

##############################################################################################################################################
#Create Tree
decisionTree=create_decision_tree(dataIris,xDataIris,yDataIris,maxDepth,trainPercShareIris,randomShuffle,randomState)
#Save results
resultEx1=update_result_3outputs(decisionTree[0],decisionTree[1]*100,decisionTree[2]*100,decisionTree[3],resultEx1)
#Plot results
logarithmic=False
create_plot(resultEx1,logarithmic)

### Trees

In [ ]:
show_trees(resultEx1)

## Exercise 2: Decision Trees  
### Explore the Parameter **Max Depth** with **Numbers Recognition** Dataset

In [ ]:
###################################################### Specify data ##########################################################################

dataNumbers=pd.read_excel('Numbers-700.xlsx')
xDataNumbers=[str(i) for i in list(range(1,785))] #Generates a list ['1', '2', '3', ..., '783', '784'], these are the column names
yDataNumbers='Number'

In [ ]:
####################################################### For profis ###########################################################################

trainPercShareNumbers=90 #Number between 0 and 100
randomShuffle=False #True (or False)
randomState=1 #None or Number

In [ ]:
#################################################### Specify parameters ######################################################################

maxDepth=1 #Number

##############################################################################################################################################

#Create Tree
decisionTree=create_decision_tree(dataNumbers,xDataNumbers,yDataNumbers,maxDepth,trainPercShareNumbers,randomShuffle,randomState)
#Save results
resultEx2=update_result_3outputs(decisionTree[0],decisionTree[1]*100,decisionTree[2]*100,decisionTree[3],resultEx2)
#Plot accuracy results
logarithmic=False
create_plot(resultEx2,logarithmic)
#Plot confusion matrix
sns.heatmap(confusion_matrix(decisionTree[4][3], decisionTree[3].predict(decisionTree[4][2])),cmap="Blues",annot=True,fmt="d").set(xlabel='Predicted Y-Data', ylabel='Real Y-Data')
plt.show()

### Trees

In [ ]:
show_trees(resultEx2)

## Exercise 3.1: Neural Networks 
### Explore the Parameter **Learning Rate** with **Numbers Recognition** Dataset

In [ ]:
####################################################### For profis ###########################################################################

hiddenLayers31=1 #Number
neurons=100 #Number
activationFkt='logistic' #‘identity’, ‘logistic’, ‘tanh’ or ‘relu’
solvingAlgo='sgd' #‘lbfgs’, ‘sgd’, ‘adam’
maxEpochs=200 #Number (max number of iterations)
trainPercShareNumbers=90 #Number between 0 and 100
randomShuffle=False #True (or False)
randomState=1 #None or Number
exactHiddenLayerStructure=False #False (same NumNeurons in each HidLay) or from type [100,10,40] ([NumNeur in HidLay1, NumNeur in HidLay2, ... ])

In [ ]:
#################################################### Specify parameters ######################################################################

learningRate=0.0001 #Number

##############################################################################################################################################

#Create Neural Network
neuralNetwork=create_neural_network(dataNumbers,xDataNumbers,yDataNumbers,learningRate,hiddenLayers31,neurons,activationFkt,solvingAlgo,maxEpochs,trainPercShareNumbers,randomShuffle,randomState,exactHiddenLayerStructure)
#Save results
resultEx31=update_result_2outputs(neuralNetwork[0],neuralNetwork[2]*100,neuralNetwork[3]*100,resultEx31)
#Plot results
logarithmic=True
create_plot(resultEx31,logarithmic)    

## Exercise 3.2 (Optional): Neural Networks 
### Explore the Parameter **Hidden Layers** with **Numbers Recognition** Dataset

In [ ]:
####################################################### For profis ###########################################################################

neurons=100 #Number
activationFkt='logistic' #‘identity’, ‘logistic’, ‘tanh’ or ‘relu’
solvingAlgo='sgd' #‘lbfgs’, ‘sgd’, ‘adam’
maxEpochs=200 #Number (max number of iterations)
trainPercShareNumbers=90 #Number between 0 and 100
randomShuffle=False #True (or False)
randomState=1 #None or Number
exactHiddenLayerStructure=False #False (same NumNeurons in each HidLay) or from type [100,10,40] ([NumNeur in HidLay1, NumNeur in HidLay2, ... ])

In [ ]:
#################################################### Specify parameters ######################################################################

learningRate=0.0001 #Number
hiddenLayers=1 #Number

##############################################################################################################################################

#Create Neural Network
neuralNetwork=create_neural_network(dataNumbers,xDataNumbers,yDataNumbers,learningRate,hiddenLayers,neurons,activationFkt,solvingAlgo,maxEpochs,trainPercShareNumbers,randomShuffle,randomState,exactHiddenLayerStructure)
#Save results
resultEx32=update_result_2outputs(neuralNetwork[1],neuralNetwork[2]*100,neuralNetwork[3]*100,resultEx32)
#Plot results
logarithmic=False
create_plot(resultEx32,logarithmic) 

## For more details go to: https://scikit-learn.org/stable/